In [7]:
import orjson
import genson
import natsort
import polars
import rich
from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig
import pyiceberg
import datetime

In [ ]:
# --- Sinks config ---
data_index_catalog_config = S3TablesCatalogConfig(
    region="ap-southeast-2",
    arn="arn:aws:s3tables:ap-southeast-2:704910415367:bucket/imos-data-inventory",
)

table = IcebergTableConfig(
    catalog_config=data_index_catalog_config,
    namespace="inventory",
    table_name=f"live",
).load()

In [3]:
table.schema

<bound method Table.schema of live(
  1: bucket: required string (The general purpose bucket name.),
  2: key: required string (The object key name (or key) that uniquely identifies the object in the bucket.),
  3: sequence_number: required string (The sequence number, which is an ordinal that's included in the records for a given object. To order records of the same bucket and key, you can sort on sequence_number.),
  4: version_id: optional string (The object's version ID. Amazon S3 assigns a version number to objects added to the bucket.),
  5: is_delete_marker: optional boolean (The object's delete marker status. True if the object is a delete marker.),
  6: size: optional long (The object size in bytes. If is_delete_marker is True, the size is 0.),
  7: last_modified_date: optional timestamp (The object creation date or the last modified date, whichever is the latest.),
  8: e_tag: optional string (The entity tag (ETag), which is a hash of the object contents.),
  9: storage_class

In [26]:
def get_lookback_timestamp(time_delta: datetime.timedelta) -> str:
    return datetime.datetime.combine(
        date=datetime.date.today() - time_delta, 
        time=datetime.time.min,
    ).isoformat()

get_lookback_timestamp(time_delta=datetime.timedelta(days=10))

'2026-06-21T00:00:00'

In [27]:
(
    table.scan(
        row_filter=f"last_modified_date >= '{get_lookback_timestamp(time_delta=datetime.timedelta(days=10))}'"
    )
    .to_polars()
)

bucket,key,sequence_number,version_id,is_delete_marker,size,last_modified_date,e_tag,storage_class,is_multipart,encryption_status,is_bucket_key_enabled,kms_key_arn,checksum_algorithm,object_tags,user_metadata,facility,year
str,str,str,str,bool,i64,datetime[μs],str,str,bool,str,bool,str,str,list[struct[2]],list[struct[2]],str,str
"""imos-data""","""IMOS/SRS/SST/ghrsst/L3U-S/snpp…","""80ea37d0b96f8b91a27f8304ec92f7…","""b4.a1YAn0eOwkoghm9apbKSMpm6ouK…",false,1672871,2026-06-21 11:53:30,"""af92080e3b18a37a71e86c891284b8…","""STANDARD""",false,"""SSE-S3""",false,null,"""CRC32""",[],[],"""SRS""","""2026"""
"""imos-data""","""IMOS/SRS/SST/ghrsst/L3U-S/snpp…","""80ea37d0b95d4ccf116cb564c5271e…","""hZ_eOh_4gLhO0oA0itcFS3k9UecD.7…",false,2521621,2026-06-21 11:53:30,"""692e78eb029ffcbe11295ba41ef514…","""STANDARD""",false,"""SSE-S3""",false,null,"""CRC32""",[],[],"""SRS""","""2026"""
"""imos-data""","""IMOS/SRS/SST/ghrsst/L3U-S/snpp…","""80ea37d0b9231e588a689afaf3afa0…","""SE1Kaa2kp0HqCc9zx7mO3HEAHkQO1N…",false,3606526,2026-06-21 11:53:30,"""e9623c3ee0bfb8f45040a5626d5ec6…","""STANDARD""",false,"""SSE-S3""",false,null,"""CRC32""",[],[],"""SRS""","""2026"""
"""imos-data""","""IMOS/SRS/SST/ghrsst/L3U-S/snpp…","""80ea37d0b90edf9e04447fce157502…","""b4x.RcwllUJHzrXSdzndO0aVR1ogv9…",false,2266628,2026-06-21 11:53:30,"""041ab625ed4199c6a87302a5a7d96c…","""STANDARD""",false,"""SSE-S3""",false,null,"""CRC32""",[],[],"""SRS""","""2026"""
"""imos-data""","""IMOS/SRS/SST/ghrsst/L3U-S/snpp…","""80ea37d0b8f827587b6e99b187a453…","""cwDZVS9v4p.fWym3utZlTGnBbiW1bc…",false,1661179,2026-06-21 11:53:29,"""2ca2844d076b7c249a8bc545f85c97…","""STANDARD""",false,"""SSE-S3""",false,null,"""CRC32""",[],[],"""SRS""","""2026"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""imos-data""","""IMOS/ACORN/radial/SEAL/2026/06…","""80ea3dc113ef5da045488ef2f472ef…","""vNm_tia2ftfWs3x3lebXQ9ao9Ta6sW…",false,143056,2026-06-26 00:00:20,"""0f5247da5c79f738efb541db9e9265…","""STANDARD""",false,"""SSE-S3""",false,null,"""CRC64NVME""",[],[],"""ACORN""","""2026"""
"""imos-data""","""IMOS/ACORN/radial/SEAL/2026/06…","""80ea3dcf1c32826c42713ba801f1e2…","""x2VlQ_U3mPAvUCrBl6EDae5ki6kbCM…",false,144025,2026-06-26 01:00:13,"""96d72c2db5b4c572350eb99f69913a…","""STANDARD""",false,"""SSE-S3""",false,null,"""CRC64NVME""",[],[],"""ACORN""","""2026"""
"""imos-data""","""IMOS/ACORN/radial/SEAL/2026/06…","""80ea3ddd34c4572bad7523a12cec64…","""RPo1Sl_ySWy4qYl6xdreJNQEP2R8at…",false,144178,2026-06-26 02:00:21,"""ed7bc9e7bbacef9196cecc4a4f87bf…","""STANDARD""",false,"""SSE-S3""",false,null,"""CRC64NVME""",[],[],"""ACORN""","""2026"""


In [ ]:
table.scan().to_polars()